# Streaming Layer - Live Order Events

Reads events off the volume incrementally with Auto Loader, aggregates
them into 5-minute windows with a watermark to handle late-arriving
data, and upserts the results into a gold table - so re-running or
catching up after a gap never creates duplicate rows.

## Setup

Auto Loader needs somewhere to persist the schema it infers and the
checkpoint (its bookmark of what's already been processed) - both go
in the volume so they survive across notebook restarts.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
from delta.tables import DeltaTable

SOURCE_PATH = "/Volumes/workspace/default/raw_data/streaming_events"
SCHEMA_LOCATION = "/Volumes/workspace/default/raw_data/_schemas/streaming_events"
CHECKPOINT_LOCATION = "/Volumes/workspace/default/raw_data/_checkpoints/streaming_events"
TARGET_TABLE = "workspace.gold.streaming_sales_pulse"

# defining this up front rather than inferring on every run - keeps the
# stream from breaking if a batch happens to arrive with a weird value
event_schema = StructType([
    StructField("event_id", StringType()),
    StructField("event_type", StringType()),
    StructField("customer_id", StringType()),
    StructField("product_category", StringType()),
    StructField("amount", DoubleType()),
    StructField("event_timestamp", TimestampType()),
])

## Read the stream with Auto Loader

`cloudFiles.schemaEvolutionMode` set to "rescue" means if a future
event ever shows up with an unexpected field, it lands in a
`_rescued_data` column instead of crashing the whole stream - real
pipelines see schema drift eventually, and this is how you survive it.

In [0]:
raw_events = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .schema(event_schema)
    .load(SOURCE_PATH)
)

## Windowed aggregation with watermarking

Watermark tells Spark "stop waiting for events older than 10 minutes
behind the newest one we've seen" - without it, the stream would have
to hold state forever to handle events that never show up late.

In [0]:
sales_pulse = (
    raw_events
    .withWatermark("event_timestamp", "10 minutes")
    .groupBy(
        F.window("event_timestamp", "5 minutes").alias("event_window"),
        "product_category",
        "event_type",
    )
    .agg(
        F.count("event_id").alias("event_count"),
        F.sum("amount").alias("total_amount"),
    )
    .select(
        F.col("event_window.start").alias("window_start"),
        F.col("event_window.end").alias("window_end"),
        "product_category",
        "event_type",
        "event_count",
        "total_amount",
    )
)

## Write with an upsert (foreachBatch + MERGE)

A plain streaming write would just append every micro-batch, so
re-running or reprocessing the same window twice would double-count
it. foreachBatch lets us run a MERGE instead - matching on the window
and category, so re-processing the same data updates the existing row
rather than duplicating it.

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
        window_start TIMESTAMP,
        window_end TIMESTAMP,
        product_category STRING,
        event_type STRING,
        event_count LONG,
        total_amount DOUBLE
    ) USING DELTA
""")


def upsert_batch(batch_df, batch_id: int) -> None:
    target = DeltaTable.forName(spark, TARGET_TABLE)

    (
        target.alias("t")
        .merge(
            batch_df.alias("s"),
            "t.window_start = s.window_start AND t.product_category = s.product_category AND t.event_type = s.event_type",
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"batch {batch_id}: merged {batch_df.count()} rows")


query = (
    sales_pulse.writeStream
    .outputMode("update")
    .foreachBatch(upsert_batch)
    .option("checkpointLocation", CHECKPOINT_LOCATION)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

## Verify

In [0]:
display(spark.sql(f"SELECT * FROM {TARGET_TABLE} ORDER BY window_start DESC LIMIT 20"))